# Which cell is it?

A hospital biobank wants a classifier that labels the cell types in single-cell data from its stored
blood samples. Its samples are **cryopreserved**, **total PBMC**, from **patients the model has
never seen**. We have data from two research labs. **How good will the classifier be in the
biobank?**

Before a model is used, the only way to answer is an estimate made from the training data. Here we
are lucky: the biobank has labelled some of its cells. So we can do what is rarely possible - compare
every estimate with the real performance, and see which one told the truth.

## How this works

Ten tasks. Each asks you to **decide** something - which split, which cells to compare, which genes
to trust - then run it with a small toolkit, then say what it means. Your code will be short. The
thinking is the work.

Where you see **Task**, the cell below it is yours; the comment gives a question to think about and
the function to use. Where you see **Your conclusion**, write one sentence. If you get stuck, skip the
task and come back.

**In Google Colab:** use a Python CPU runtime and run the setup cells first. The CSV files load directly from GitHub. Use **File → Save a copy in Drive** to keep your work.

[Lab handout](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-05-single-cells/notebooks/session_05/S5_lab_handout.md) · [Data dictionary](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-05-single-cells/notebooks/session_05/data/pbmc_DICTIONARY.md) · [Training CSV](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/session-05-single-cells/notebooks/session_05/data/pbmc_train.csv) · [Biobank CSV](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/session-05-single-cells/notebooks/session_05/data/pbmc_external.csv).


## Setup

Run the next three cells.

In [ ]:
%pip install -q numpy pandas scikit-learn


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, recall_score, roc_auc_score
pd.set_option("display.width", 200)

df = pd.read_csv("https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/session-05-single-cells/notebooks/session_05/data/pbmc_train.csv")              # the two research labs
ext = pd.read_csv("https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/session-05-single-cells/notebooks/session_05/data/pbmc_external.csv")          # the biobank's labelled cells
META = ["cell_id", "donor_id", "sex", "age", "condition", "lab", "chemistry", "processing",
        "sorting", "n_counts", "n_genes", "pct_mito"]
TARGET = "cell_type"
GENES = [c for c in df.columns if c not in META + [TARGET]]
TYPES = ["CD4 naive T", "CD4 memory T", "CD8 T", "NK", "B", "CD14 monocyte", "CD16 monocyte"]
print("training:", df.shape[0], "cells from", df.donor_id.nunique(), "donors |",
      "biobank:", ext.shape[0], "cells from", ext.donor_id.nunique(), "donors |", len(GENES), "genes")

**The score is balanced accuracy**: the mean of the seven recalls - for each cell type, the share
of its cells the model gets right. Every cell type counts the same, however common it is, and the
score does not change when the mix of cell types changes. That matters: the biobank's mix is not the
training mix.

In [ ]:
from sklearn.base import BaseEstimator
from sklearn.model_selection import BaseCrossValidator

cv_random = StratifiedKFold(5, shuffle=True, random_state=0)        # cells shuffled into 5 folds
cv_donor = StratifiedGroupKFold(5, shuffle=True, random_state=0)    # 5 folds that keep each group whole
MODELS = ["logistic regression", "random forest", "k-NN"]


def make_models() -> dict[str, BaseEstimator]:
    """Return a fresh, untrained copy of each of the three models, keyed by name."""
    # Logistic regression: genes are standardised first, so no gene dominates by its scale alone
    logistic = Pipeline([
        ("scale", StandardScaler()),
        ("m", LogisticRegression(max_iter=3000)),
    ])
    # Random forest: trees do not care about scale, so no standardising step
    forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=0)
    # k-NN: distances depend on scale, so genes are standardised first
    knn = Pipeline([
        ("scale", StandardScaler()),
        ("m", KNeighborsClassifier(n_neighbors=15)),
    ])
    return {"logistic regression": logistic, "random forest": forest, "k-NN": knn}


def in_sample(features: list[str], models: list[str] = MODELS) -> pd.Series:
    """Balanced accuracy of each model on the very cells it was trained on."""
    scores = {}
    for name in models:
        # Train a fresh model on all training cells
        model = make_models()[name]
        model.fit(df[features], df[TARGET])
        # Predict the same cells it has just seen
        predicted = model.predict(df[features])
        # Compare with the true labels
        scores[name] = balanced_accuracy_score(df[TARGET], predicted)
    return pd.Series(scores, name="in-sample").round(3)


def predictions(features: list[str], cv: BaseCrossValidator, groups: pd.Series | None = None,
                model: str = "logistic regression") -> np.ndarray:
    """Cross-validated prediction for every training cell.

    Each cell is predicted by a model that did not see that cell's fold during training.
    With `groups` (e.g. donor IDs) and a group-aware `cv`, all cells of a group stay in one fold.
    """
    return cross_val_predict(make_models()[model], df[features], df[TARGET], cv=cv, groups=groups)


def evaluate(features: list[str], cv: BaseCrossValidator, groups: pd.Series | None = None,
             models: list[str] = MODELS) -> pd.Series:
    """Cross-validated balanced accuracy of each model, with the folds given by `cv` and `groups`."""
    scores = {}
    for name in models:
        # Get the out-of-fold prediction for every training cell
        predicted = predictions(features, cv, groups, name)
        # Score them against the true labels
        scores[name] = balanced_accuracy_score(df[TARGET], predicted)
    return pd.Series(scores, name="cross-validated").round(3)


def biobank_predictions(features: list[str], model: str = "logistic regression") -> np.ndarray:
    """Train on all training cells, then predict every biobank cell."""
    trained = make_models()[model]
    trained.fit(df[features], df[TARGET])
    return trained.predict(ext[features])


def biobank(features: list[str], models: list[str] = MODELS) -> pd.Series:
    """Balanced accuracy of each model in the biobank, trained on all training cells."""
    scores = {}
    for name in models:
        # Train on the research labs, predict the biobank
        predicted = biobank_predictions(features, name)
        # Score against the biobank's own labels
        scores[name] = balanced_accuracy_score(ext[TARGET], predicted)
    return pd.Series(scores, name="biobank").round(3)


def recall_table(true: pd.Series | np.ndarray, predicted: pd.Series | np.ndarray) -> pd.DataFrame:
    """Recall and number of cells for each cell type, in the fixed order of TYPES."""
    # Drop any index so `true` lines up with `predicted` by position
    true = pd.Series(np.asarray(true))
    # Recall per cell type: the share of that type's cells that were predicted correctly
    recalls = recall_score(true, predicted, labels=TYPES, average=None).round(2)
    # Number of cells of each type (0 if a type is absent)
    counts = true.value_counts().reindex(TYPES).fillna(0).astype(int).values
    return pd.DataFrame({"recall": recalls, "cells": counts}, index=TYPES)


def lab_difference(cells: pd.DataFrame) -> pd.Series:
    """Mean expression in lab B minus lab A for each gene, largest difference (either sign) first."""
    # Mean of every gene within each lab: one row per lab, one column per gene
    means = cells.groupby("lab")[GENES].mean()
    # Difference per gene; positive means higher in lab B
    diff = means.loc["B"] - means.loc["A"]
    # Sort by size of the difference, ignoring its sign
    order = diff.abs().sort_values(ascending=False).index
    return diff.reindex(order).round(2)


def top_genes(cell_type: str, features: list[str], n: int = 10) -> pd.Series:
    """The `n` genes with the largest positive logistic-regression weight for `cell_type`."""
    # Train the logistic model on all training cells and take the classifier out of the pipeline
    pipeline = make_models()["logistic regression"]
    pipeline.fit(df[features], df[TARGET])
    lr = pipeline.named_steps["m"]
    # coef_ has one row of weights per cell type; pick the row for this one
    row = list(lr.classes_).index(cell_type)
    weights = pd.Series(lr.coef_[row], index=features)
    # Genes are standardised, so the weights are comparable; largest first
    return weights.sort_values(ascending=False).head(n).round(2)


def sle_auc(cv: BaseCrossValidator, groups: pd.Series | None = None) -> pd.Series:
    """Cross-validated AUC for predicting SLE from all genes, scored over cells and over donors."""
    # Target: 1 if the cell's donor has SLE, 0 if healthy
    y = (df.condition == "SLE").astype(int)
    # Out-of-fold probability of SLE for every cell (column 1 of predict_proba is the SLE class)
    proba = cross_val_predict(make_models()["logistic regression"], df[GENES], y,
                              cv=cv, groups=groups, method="predict_proba")
    p = proba[:, 1]
    # Average label and probability per donor: one row per person
    per_cell = pd.DataFrame({"donor": df.donor_id, "y": y, "p": p})
    per_donor = per_cell.groupby("donor").mean()
    # AUC over cells counts every cell; AUC over donors counts every person once
    return pd.Series({
        "AUC over cells": roc_auc_score(y, p),
        "AUC over donors": roc_auc_score(per_donor.y, per_donor.p),
    }).round(3)

print("Toolkit loaded.")

**The toolkit**

| Function | What it gives you |
|---|---|
| `in_sample(features)` | balanced accuracy of the three models on the very cells they were trained on |
| `evaluate(features, cv, groups=None)` | balanced accuracy of the three models, cross-validated with the folds you choose |
| `biobank(features)` | balanced accuracy of the three models in the biobank - trained on all training cells |
| `predictions(features, cv, groups=None)` | the logistic model's cross-validated prediction for every training cell |
| `biobank_predictions(features)` | the logistic model's prediction for every biobank cell |
| `recall_table(true, predicted)` | recall and number of cells, for each cell type |
| `lab_difference(cells)` | mean expression in lab B minus lab A, for each gene, largest first |
| `top_genes(cell_type, features)` | the genes the logistic model leans on most for that cell type |
| `sle_auc(cv, groups=None)` | for block E: AUC over cells and over donors |

`features` is a list of gene names, for example `GENES`. `groups` is a column that says which cells
belong together.

## A · The data

Before any model: where did the training cells come from?

### Task 1 · Where do the cells come from?
Make one table that shows which cell types each lab contributed. What stands out? Which metadata
column explains it?

In [ ]:
# TASK 1
# Think: which two columns, crossed, would show this?
# Use:   pd.crosstab(...)    and, for the explanation, df.groupby("lab")[[...]].first()



## B · How good is it? Every estimate

Start with the obvious model: all 40 genes. Measure it every way you can from the training data -
then compare with the biobank.

### Task 2 · In-sample, and random folds
Score the three models on the cells they were trained on, then with `cv_random`. What is an in-sample
score worth?

In [ ]:
# TASK 2
# Think: which cells is each number computed on - cells the model has seen, or not?
# Use:   in_sample(GENES)    evaluate(GENES, cv_random)



### Task 3 · Folds that match the question
The biobank's patients are people the model has never seen. Choose folds that match that, and score
the three models again.

In [ ]:
# TASK 3
# Think: who will the model meet in the biobank that it has never seen?
#        Which column says who a cell belongs to?
# Use:   evaluate(GENES, cv=..., groups=...)



### Task 4 · The real performance
Score the three models in the biobank. Put every estimate - in-sample, random folds, donor folds -
next to the biobank in one table. Which estimate came closest? Was any of them close enough?

In [ ]:
# TASK 4
# Think: how would you lay the four numbers side by side, for each model?
# Use:   biobank(GENES)    pd.DataFrame({...})



**Your conclusion.** One sentence: what does this block show?

## C · Why was even the best estimate wrong?

Donor folds test on new people, as the biobank will. Yet they were still too optimistic. So something
in the training data must make the problem look easier than it is - something that does not change
when you hold out a donor.

### Task 5 · Which genes differ between the labs?
Use `lab_difference` on all the cells. Then use it on **one cell type of your choice**. Which genes
still differ when the cell type is the same? Look them up in the data dictionary.

End the task with your own list of genes you would not trust, as `suspect = [...]`.

In [ ]:
# TASK 5
# Think: lab B loaded different cell types. If you compare one cell type only,
#        which differences can no longer come from that?
# Use:   lab_difference(df)    lab_difference(df[df.cell_type == "..."])
#        suspect = [...]



*What could make these genes differ between the labs, when the cell type is the same? Write it below.*

*Your answer:*

### Task 6 · What does the model lean on?
List the genes the logistic model leans on most for "CD4 memory T". Are any of your suspects among
them? If the model had learned memory-T biology, what would you expect at the top?

In [ ]:
# TASK 6
# Think: which kind of genes should a memory-T classifier lean on?
# Use:   top_genes("CD4 memory T", GENES)      then compare with suspect



**Your conclusion.** One sentence: what does this block show?

## D · Remove it, and measure again

If the handling genes were the problem, a model without them should be both better in the biobank
and more honestly measured by cross-validation.

### Task 7 · The same table, without your suspects
Make the list of genes without your suspects. Build the same table as in Task 4 for it. Compare the two
tables: which model is better in the biobank? For which model did the donor folds tell the truth?

In [ ]:
# TASK 7
# Think: which genes do you leave out, given Tasks 5 and 6?
# Use:   honest = [g for g in GENES if g not in suspect]
#        in_sample(honest)    evaluate(honest, ...)    biobank(honest)



### Task 8 · What did each estimate promise, cell type by cell type?
An average can hide a lot. For the logistic model, with and without your suspects: what recall did the
donor folds promise for **naive** and **memory** CD4 cells, and what did the biobank deliver?

In [ ]:
# TASK 8
# Think: the same two cell types, two gene lists, two kinds of number.
# Use:   recall_table(df[TARGET], predictions(..., cv_donor, groups=df["donor_id"]))
#        recall_table(ext[TARGET], biobank_predictions(...))



**Your conclusion.** One sentence: what does this block show?

## E · A different question

Same cells, a new label. **SLE - systemic lupus erythematosus - is an autoimmune disease**: the
immune system attacks the body's own tissues. Eight of the 16 donors have SLE. The question is
whether a donor has SLE, judged from single cells. Every cell of a donor carries the same answer.

### Task 9 · SLE from single cells
Use `sle_auc` twice: with `cv_random`, and with folds that keep each donor whole. It gives the AUC over
cells, and over donors after averaging each donor's predictions. Which numbers would you believe - and
how many independent samples do you really have?

In [ ]:
# TASK 9
# Think: with 16 donors, 8 with SLE, how many donor folds can you make so that every fold
#        contains both kinds of donor?
# Use:   sle_auc(cv_random)    sle_auc(StratifiedGroupKFold(..., shuffle=True, random_state=0), groups=...)



*How many independent samples do you have for this question, and what does that mean for the AUC? Write it below.*

*Your answer:*

**Your conclusion.** One sentence: what does this block show?

## F · The answer

### Task 10 · How good will the classifier be in the biobank?
Answer the question we started with, in three lines:

**The number, and which model:**

**Where the model can be trusted, and where it cannot:**

**Which estimate from the training data would have told you this - before you saw the biobank?**

Then **upload this notebook to Moodle**.

In [ ]:
# scratch
